# Results Overview — `career_position` Annotation

Aggregates all submitted results from `results/register.csv` for the `annotation / career_position` task.
Models are grouped by architecture family for both **broad** (9 sectors) and **fine-grained** (~100 codes) granularities.

> Run all cells to regenerate after new results are submitted.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

REGISTER = Path("/home/tom/projects/corex_eval/results/register.csv")

TASK = "annotation"
VAR  = "career_position"

## 1. Model catalogue

In [ ]:
# (family, display_name, experiment_folder_broad, experiment_folder_fine)
CATALOGUE = [
    # Encoders
    ("Encoder", "mBERT",             "bert_finetuned_career_broad",         "bert_finetuned_career"),
    ("Encoder", "BERT-EN",           "bert_english_finetuned_career_broad", "bert_english_finetuned_career"),
    ("Encoder", "XLM-RoBERTa",      "xlmroberta_finetuned_career_broad",   "xlmroberta_finetuned_career"),
    # Open-source LLMs
    ("OSS LLM", "GPT-OSS 20B",      "lmstudio_gptoss20b_broad",            "lmstudio_gptoss20b_fine"),
    ("OSS LLM", "Llama 3.3 70B",    "lmstudio_llama33_70b_broad",          "lmstudio_llama33_70b_fine"),
    ("OSS LLM", "Qwen3 80B",        "lmstudio_qwen3_80b_broad",            "lmstudio_qwen3_80b_fine"),
    # Closed-source LLMs
    ("Closed",  "GPT-4o",           "gpt4o_broad",                         "gpt4o_fine"),
    ("Closed",  "GPT-5.3",          "gpt53_broad",                         "gpt53_fine"),
    ("Closed",  "Claude Opus 4.6",  "claude_opus46_broad",                 "claude_opus46_fine"),
    ("Closed",  "Gemini 2.5 Flash", "gemini25flash_broad",                 "gemini25flash_fine"),
    # RAG
    ("RAG",     "RAG-Embeddings",   "rag_embeddings_broad",                "rag_embeddings_fine"),
    ("RAG",     "RAG-WebSearch",    "rag_websearch_broad",                 "rag_websearch_fine"),
]

FAMILY_ORDER  = ["Encoder", "OSS LLM", "Closed", "RAG"]
FAMILY_COLORS = {
    "Encoder": "#4C72B0",
    "OSS LLM": "#55A868",
    "Closed":  "#C44E52",
    "RAG":     "#DD8452",
}

print(f"{len(CATALOGUE)} experiments in catalogue")

## 2. Load register

In [ ]:
reg = pd.read_csv(REGISTER)
reg = reg[(reg["task"] == TASK) & (reg["variable"] == VAR)].copy()

def _folder(path):
    parts = str(path).replace("\\", "/").split("/")
    return parts[-2] if len(parts) >= 2 else str(path)

reg["exp_folder"] = reg["experiment_path"].apply(_folder)

for col in ["accuracy", "macro_f1", "weighted_f1"]:
    reg[col] = pd.to_numeric(reg[col], errors="coerce")

# Keep best accuracy per experiment folder, drop broken/zero-accuracy rows
reg_best = (
    reg[reg["accuracy"] > 0]
    .sort_values("accuracy", ascending=False)
    .drop_duplicates(subset="exp_folder", keep="first")
    .set_index("exp_folder")
)

print(f"Valid rows: {len(reg_best)}")
print("Folders:", sorted(reg_best.index.tolist()))

## 3. Build results tables

In [ ]:
def build_df(granularity):
    records = []
    for family, name, broad_folder, fine_folder in CATALOGUE:
        folder = broad_folder if granularity == "broad" else fine_folder
        if folder in reg_best.index:
            row = reg_best.loc[folder]
            records.append({
                "Family":      family,
                "Model":       name,
                "Accuracy":    round(row["accuracy"],    4),
                "Macro F1":    round(row["macro_f1"],    4) if pd.notna(row["macro_f1"])    else None,
                "Weighted F1": round(row["weighted_f1"], 4) if pd.notna(row["weighted_f1"]) else None,
            })
        else:
            records.append({"Family": family, "Model": name,
                            "Accuracy": None, "Macro F1": None, "Weighted F1": None})
    df = pd.DataFrame(records)
    df["Family"] = pd.Categorical(df["Family"], categories=FAMILY_ORDER, ordered=True)
    return df.sort_values(["Family", "Model"]).reset_index(drop=True)

df_broad = build_df("broad")
df_fine  = build_df("fine")

## 4. Broad sector results (9 classes)

In [ ]:
def fmt_table(df):
    d = df.copy()
    for col in ["Accuracy", "Macro F1", "Weighted F1"]:
        d[col] = d[col].apply(lambda x: f"{x:.4f}" if pd.notna(x) else "TBD")
    return d.style.hide(axis="index")

fmt_table(df_broad)

In [ ]:
def plot_results(df, title):
    has = df[df["Accuracy"].notna()].copy()
    tbd = df[df["Accuracy"].isna()]["Model"].tolist()

    x     = np.arange(len(has))
    w     = 0.26
    metrics = [("Accuracy", 1.0), ("Macro F1", 0.6), ("Weighted F1", 0.35)]
    offsets = [-w, 0, w]
    bar_colors = [FAMILY_COLORS[f] for f in has["Family"]]

    fig, ax = plt.subplots(figsize=(max(10, len(has) * 1.0), 5))

    for (metric, alpha), offset in zip(metrics, offsets):
        vals = has[metric].values
        bars = ax.bar(
            x + offset, vals, w,
            color=[c + f"{int(255*alpha):02x}" for c in bar_colors],
            edgecolor="white", linewidth=0.5, label=metric,
        )
        for bar, val in zip(bars, vals):
            if pd.notna(val):
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                        f"{val:.2f}", ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(
        [f"{r.Model}\n({r.Family})" for _, r in has.iterrows()],
        fontsize=9, rotation=30, ha="right",
    )
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title(title, fontsize=13, fontweight="bold")
    ax.grid(axis="y", alpha=0.3)
    ax.axhline(0, color="grey", linewidth=0.5)

    family_patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
    metric_patches = [
        mpatches.Patch(color="grey", alpha=1.0,  label="Accuracy"),
        mpatches.Patch(color="grey", alpha=0.6,  label="Macro F1"),
        mpatches.Patch(color="grey", alpha=0.35, label="Weighted F1"),
    ]
    ax.legend(handles=family_patches + metric_patches, fontsize=8, ncol=2, loc="upper right")

    if tbd:
        ax.text(0.01, 0.02, "TBD: " + ", ".join(tbd),
                transform=ax.transAxes, fontsize=8, color="grey", va="bottom")

    plt.tight_layout()
    plt.show()

plot_results(df_broad, "career_position — Broad sectors (9 classes)")

## 5. Fine-grained results (~100 codes)

In [ ]:
fmt_table(df_fine)

In [ ]:
plot_results(df_fine, "career_position — Fine-grained (~100 codes)")

## 6. Broad vs fine — accuracy side-by-side

In [ ]:
merged = (
    df_broad[["Family", "Model", "Accuracy", "Macro F1"]]
    .rename(columns={"Accuracy": "Acc (broad)", "Macro F1": "MF1 (broad)"})
    .merge(
        df_fine[["Model", "Accuracy", "Macro F1"]]
        .rename(columns={"Accuracy": "Acc (fine)", "Macro F1": "MF1 (fine)"}),
        on="Model",
    )
)
has = merged[merged["Acc (broad)"].notna() | merged["Acc (fine)"].notna()].copy()

x = np.arange(len(has))
w = 0.35
bar_colors = [FAMILY_COLORS[f] for f in has["Family"]]

fig, ax = plt.subplots(figsize=(max(10, len(has) * 1.0), 5))
b1 = ax.bar(x - w/2, has["Acc (broad)"].fillna(0), w, color=bar_colors, alpha=1.0, edgecolor="white", label="Broad")
b2 = ax.bar(x + w/2, has["Acc (fine)"].fillna(0),  w, color=bar_colors, alpha=0.45, edgecolor="white", label="Fine")

for bars, col in [(b1, "Acc (broad)"), (b2, "Acc (fine)")]:
    for bar, (_, row) in zip(bars, has.iterrows()):
        val = row[col]
        if pd.notna(val) and val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                    f"{val:.2f}", ha="center", va="bottom", fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(
    [f"{r.Model}\n({r.Family})" for _, r in has.iterrows()],
    fontsize=9, rotation=30, ha="right",
)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Accuracy")
ax.set_title("Accuracy: Broad vs Fine-grained", fontsize=13, fontweight="bold")
ax.grid(axis="y", alpha=0.3)

family_patches = [mpatches.Patch(color=c, label=f) for f, c in FAMILY_COLORS.items()]
gran_patches = [
    mpatches.Patch(color="grey", alpha=1.0,  label="Broad (solid)"),
    mpatches.Patch(color="grey", alpha=0.45, label="Fine (faded)"),
]
ax.legend(handles=family_patches + gran_patches, fontsize=8, ncol=2, loc="upper right")

tbd = merged[merged["Acc (broad)"].isna() & merged["Acc (fine)"].isna()]["Model"].tolist()
if tbd:
    ax.text(0.01, 0.02, "TBD: " + ", ".join(tbd),
            transform=ax.transAxes, fontsize=8, color="grey", va="bottom")

plt.tight_layout()
plt.show()